# ELIZA kontra LLM — gdzie jest „rozumienie"?

**Podstawy AI · Laboratorium 2**

Na poprzednich zajęciach zbudowałeś ELIZĘ i znalazłeś miejsca, w których
jej złudzenie pęka. Dziś te same zdania trafią do dużego modelu językowego
(LLM) i sprawdzimy: **które pęknięcia LLM naprawia, a które zostają?**

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Jak rozmawia się z LLM przez API | czytasz (5 min) |
| 2. Przygotowanie | klucz, klient, test połączenia |
| 3. Twoja ELIZA wraca | wklejasz kod z Lab 1 |
| 4. Ta sama rozmowa, dwa silniki | Zadanie 8 |
| 5. Łamiące zdania — druga runda | Zadanie 9 |
| 6. Pamięć to lista wiadomości | Zadanie 10 |
| 7. Rola to tekst | Zadanie 11 |
| 8. Refleksja | Zadanie 12 |

**Potrzebujesz:** klucza Gemini API (instrukcję dostałeś przed zajęciami)
i notebooka z Lab 1.

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 1. Jak rozmawia się z LLM przez API

**Czym jest LLM (w pięciu zdaniach).** Duży model językowy to sieć neuronowa
wytrenowana na ogromnych zbiorach tekstu do jednego zadania: przewidywania,
jaki fragment tekstu (*token*) powinien pojawić się dalej. Odpowiedź powstaje
token po tokenie — model za każdym razem wybiera kolejny fragment, a wybór
jest częściowo losowy (*próbkowanie*). Nie ma tu reguł pisanych ręcznie jak
w ELIZIE: wszystko, co model „umie", jest zapisane w miliardach wyuczonych
parametrów. To, czy i w jakim sensie model *rozumie* tekst, jest przedmiotem
sporu — i właśnie dlatego porównujemy go dziś z ELIZĄ.

**Zapytanie do API.** Rozmawiamy z modelem, wysyłając mu **listę wiadomości**.
Każda wiadomość ma rolę:

| Rola | Kto mówi | Przykład |
|---|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać | „Jesteś terapeutą. Odpowiadaj krótko." |
| `user` | użytkownik | „Czuję się samotny." |
| `assistant` | wcześniejsze odpowiedzi modelu | „Od kiedy tak się czujesz?" |

```python
messages = [
    {"role": "system", "content": "Jesteś terapeutą..."},
    {"role": "user",   "content": "Czuję się samotny."},
]
```

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 10 okaże się ważniejszy, niż wygląda.

**Biblioteka.** Użyjemy biblioteki `openai`. Mimo nazwy nie łączymy się
z OpenAI: Gemini udostępnia *zgodny* interfejs, więc wystarczy podać inny
adres serwera (`base_url`) i swój klucz.

## 2. Przygotowanie

Klucz podajesz przez `getpass` — nie pojawi się na ekranie ani w zapisanym
notebooku. **Nigdy nie wklejaj klucza wprost do kodu**: notebook oddajesz
prowadzącemu, a klucz to dostęp do Twojego konta.

Jeśli pracujesz w Google Colab, możesz zamiast tego zapisać klucz
w *Secrets* (ikona klucza po lewej) pod nazwą `GEMINI_API_KEY`.

In [1]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu -> szybciej i taniej;
                                  # jeśli dostajesz błąd 400 dotyczący reasoning_effort, ustaw None
print("Klient gotowy. Model:", MODEL)

Wklej klucz Gemini API: ··········
Klient gotowy. Model: gemini-3.5-flash-lite


In [2]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Poniższa funkcja `czat` wysyła listę wiadomości i zwraca tekst odpowiedzi.
Obsługuje też błąd **429** (przekroczony limit zapytań na minutę) — wtedy
odczekuje chwilę i próbuje ponownie. Nie musisz jej modyfikować.

In [3]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Rozumiem, to bardzo trudny stan. Chcesz o tym opowiedzieć?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 3. Twoja ELIZA wraca

Wklej poniżej **swoje** komórki z Lab 1: `normalizuj`, `odbicia`, `odbij`,
`reguly` (z regułą awaryjną) i `odpowiedz`. Porównujemy **Twoją** ELIZĘ.

Jeśli Twoja wersja nie działa, uruchom komórkę referencyjną pod spodem.

In [5]:
import random
import re

def normalizuj(tekst):
    tekst = tekst.lower()
    tekst = re.sub(r"[^\w\s]", "", tekst)  # usuń wszystko poza literami, cyframi i spacjami
    return tekst.strip()

odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    "mój": "twój",
    # Zadanie 1: dopisz min. 6 par
    "moje": "twoje",
    "mam": "masz",
    "mi": "ci",
    "ja": "ty",
    "moim": "twoim",
    "moich": "twoich",
    # kierunek odwrotny (terapeuta -> pacjent)
    "ci": "mi",
    "ty": "ja",
}

def odbij(fragment):
    slowa = fragment.lower().split()
    # Zadanie 1: zamień każde słowo na jego odbicie, jeśli jest w słowniku, w przeciwnym razie zostaw bez zmian.
    return " ".join(odbicia.get(s, s) for s in slowa)

reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    (r"boję się (.*)",
        ["Dlaczego boisz się {0}?", "Od kiedy boisz się {0}?"]),
    (r"chciałbym (.*)",
        ["Dlaczego chciałbyś {0}?", "Co by się zmieniło, gdybyś mógł {0}?"]),
    (r"chcę (.*)",
        ["Dlaczego chcesz {0}?", "Co ci daje to, że chcesz {0}?"]),
    (r"jestem (.*)",
        ["Od jak dawna jesteś {0}?", "Jak to jest być {0}?"]),
    (r"\b(nie wiem|nie mam pojęcia)\b",
        ["Czego dokładnie nie wiesz?", "Co sprawia, że trudno ci to powiedzieć?"]),
    (r"\b(wszyscy|nikt|zawsze|nigdy)\b",
        ["Czy naprawdę {0}?", "Czy możesz podać konkretny przykład?"]),
    (r"\b(matka|ojciec|rodzina|siostra|brat)\b",
        ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    (r"odczuwam (.*)",
        ["Dlaczego odczuwasz {0}?", "Od jak dawna czujesz {0}?"]),
    (r"muszę (.*)",
        ["Dlaczego czujesz, że to {0} musisz?", "Czy naprawdę musisz {0}?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            # Zadanie 3a: odbij zaimki w każdej przechwyconej grupie
            grupy = [odbij(g) for g in m.groups()]
            # Zadanie 3b: wylosuj szablon i wstaw grupy w miejsce {0}, {1}...
            return random.choice(szablony).format(*grupy)

    # Reguła awaryjna (Zadanie 4)
    return "Rozumiem. Opowiedz mi o tym więcej."

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
    "Szkoda ze nie zobaczylem."
]

if __name__ == "__main__":
    print(f"Gotowe. Zdań testowych: {len(TESTY)}\n")
    print("--- Test przykładowych wypowiedzi ---")
    for test in TESTY:
        print(f"Pacjent: {test}")
        print(f"ELIZA:   {odpowiedz(test)}")
        print("-" * 40)

Gotowe. Zdań testowych: 7

--- Test przykładowych wypowiedzi ---
Pacjent: Potrzebuję, żeby moja rodzina mnie słuchała.
ELIZA:   Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
----------------------------------------
Pacjent: Czuję się dzisiaj samotny.
ELIZA:   Od jak dawna czujesz się dzisiaj samotny?
----------------------------------------
Pacjent: Mój ojciec ciągle mnie krytykuje.
ELIZA:   Jak układają się te relacje?
----------------------------------------
Pacjent: Nie wiem, o co ci chodzi.
ELIZA:   Czego dokładnie nie wiesz?
----------------------------------------
Pacjent: Wszyscy mnie ignorują.
ELIZA:   Czy naprawdę wszyscy?
----------------------------------------
Pacjent: Boję się, że mnie zostawią.
ELIZA:   Dlaczego boisz się że ciebie zostawią?
----------------------------------------
Pacjent: Szkoda ze nie zobaczylem.
ELIZA:   Rozumiem. Opowiedz mi o tym więcej.
----------------------------------------


In [6]:
# === Referencyjna (uproszczona) ELIZA — tylko jeśli Twoja nie działa ===
import re, random

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {"jestem": "jesteś", "moja": "twoja", "mój": "twój", "moje": "twoje",
           "mnie": "ciebie", "mi": "ci", "ja": "ty", "ci": "mi", "tobie": "mnie"}

def odbij(fragment):
    return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

reguly = [
    (r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (r"boję się (.*)",   ["Dlaczego boisz się {0}?"]),
    (r"\b(matka|ojciec|rodzina|brat|siostra)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            return random.choice(szablony).format(*[odbij(g) for g in m.groups()])

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]
print(odpowiedz("Czuję się dzisiaj samotny."))

Od jak dawna czujesz się dzisiaj samotny?


## 4. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 8
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Pod spodem zapisz **3 konkretne różnice**, które widzisz. Nie oceniaj,
   który jest „lepszy" — opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [7]:
wyniki = []
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)
    wyniki.append((zdanie, odp_eliza, odp_llm))
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
  LLM:   To zrozumiałe, że potrzebujesz poczuć się ważny i wysłuchany przez bliskich. Co dokładnie sprawia, że masz wrażenie, iż Cię nie słyszą?

> Czuję się dzisiaj samotny.
  ELIZA: Od jak dawna czujesz się dzisiaj samotny?
  LLM:   Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być bardzo bolesne, kiedy bliska osoba nie daje Ci wsparcia, którego potrzebujesz. Co dokładnie zazwyczaj wtedy czujesz?

> Nie wiem, o co ci chodzi.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Rozumiem, że możesz czuć się zagubiony w tym momencie. O czym chciałbyś teraz porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: Co masz na myśli?
  LLM:   To musi być bardzo bolesne i samotne uczucie dla Ciebie. Chcesz o tym opowiedzieć coś więcej?

> Boję się, że mnie zostawią.
  ELIZA: Dlaczego boisz s

**Różnice (Zadanie 8):**

1. Szukanie słów vs. łapanie sensu:
   ELIZA działa tępo na sztywnych regexach – jak nie znajdzie konkretnego słowa (np. „potrzebuję”), to rzuca tekst awaryjny. LLM ogarnia kontekst całego zdania, nawet jak inaczej je sformułujesz.

2. Szablony vs. własne wymyślanie:
   ELIZA bierze Twój fragment, podmienia zaimki (ja->ty) i wkleja w gotowy szablon pytania. LLM generuje odpowiedź od zera, token po tokenie, więc brzmi to o wiele naturalniej i potrafi nawiązać do emocji.

3. Odporność na błędy:
   W ELIZIE literówka albo brak interpunkcji potrafią zepsuć całe dopasowanie. LLM radzi sobie z potocznym językiem, skrótami myślowymi i literówkami bez problemu.

## 5. Łamiące zdania — druga runda

W Zadaniu 6 znalazłeś zdania, które łamały ELIZĘ. Czy łamią też LLM?

### 🔧 Zadanie 9
1. Wklej swoje zdania łamiące z Lab 1 i uruchom porównanie.
2. **Zanim uruchomisz** komórkę z testem pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę: które typy pęknięć LLM naprawia, a które zostają?

In [9]:
przykłady_lamiace = [
    # 1. Negacja
    "Ja wcale nie mam problemu ze złością.",
    # 2. Zła gramatyka
    "Proszę, pomóż mnie dzisiaj.",
    # 3. Ironia
    "Jasne, czuję się świetnie, po prostu uwielbiam, kiedy wszystko mi się psuje.",
    # 4. Brak pamięci
    "To o niej właśnie cały czas myślę od rana."
]

for zdanie in przykłady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Ja wcale nie mam problemu ze złością.
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Rozumiem. Co sprawia, że dzisiaj o tym rozmawiamy?

> Proszę, pomóż mnie dzisiaj.
  ELIZA: Co masz na myśli?
  LLM:   Oczywiście, jestem tu dla Ciebie. O czym chciałbyś dzisiaj porozmawiać?

> Jasne, czuję się świetnie, po prostu uwielbiam, kiedy wszystko mi się psuje.
  ELIZA: Od jak dawna czujesz się świetnie po prostu uwielbiam kiedy wszystko ci się psuje?
  LLM:   Słyszę w Twoim głosie sporo frustracji i ironii. Co takiego dzisiaj poszło nie po Twojej myśli?

> To o niej właśnie cały czas myślę od rana.
  ELIZA: Co masz na myśli?
  LLM:   Co takiego w tej osobie sprawia, że zajmuje teraz tak wiele Twoich myśli? Jak się czujesz z tym, że wraca do Ciebie od rana?



**Moje przewidywanie (przed uruchomieniem):** …

In [8]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te słowa?
LLM: Nie wiem, jak ma na imię twój brat, ponieważ dopiero zaczynamy rozmowę. Możesz mi o nim opowiedzieć, jeśli chcesz.


**Tabela (Zadanie 9):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | ✔️ | LLM rozumie semantykę zaprzeczeń, a ELIZA szuka tylko sztywnych słów kluczowych. |
| ironia | ❌ | ✔️ | LLM potrafi wychwycić ukryty sarkazm i kontekst, podczas gdy ELIZA traktuje wypowiedź dosłownie. |
| polska gramatyka / odbicie zaimków | ❌ | ✔️ | LLM zna fleksję i polskie przypadki, ELIZA korzysta ze sztywnego słownika 1:1. |
| pamięć poprzednich zdań | ❌ | ❌ | Przy osobnych wywołaniach API model nie ma dostępu do historii poprzedniej wypowiedzi. |

## 6. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami — każde zapytanie
jest obsługiwane od zera. To, że czatboty „pamiętają", wynika z tego,
że program **odsyła modelowi całą dotychczasową rozmowę** przy każdym
nowym pytaniu.

### 🔧 Zadanie 10
a) i b) Uzupełnij funkcję `rozmowa_z_pamiecia`: po każdej turze dopisuj do
`historia` wypowiedź użytkownika (rola `user`) i odpowiedź modelu
(rola `assistant`).
c) Uruchom test z bratem Tomkiem i komórkę mierzącą rozmiar historii.
Odpowiedz pisemnie: **gdzie jest pamięć — w modelu czy w Twoim kodzie?**
Co rośnie z każdą turą i jakie to może mieć konsekwencje?

In [10]:
def rozmowa_z_pamiecia(wejascia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejascia:
        # Zadania 10a: dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})

        odp = czat(historia)

        # Zadania 10b: dopisz odpowiedź modelu do historii
        historia.append({"role": "assistant", "content": odp})

        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [11]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To z pewnością bardzo bolesne, kiedy słyszysz krytykę od kogoś bliskiego. Jak zazwyczaj na nią reagujesz?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Co takiego dokładnie doprowadziło do tej wczorajszej kłótni? Jak się wtedy czułeś?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Co dokładnie wtedy do Ciebie powiedział, że tak Cię to dotknęło?



In [12]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")

print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To z pewnością bardzo bolesne, kiedy słyszysz krytykę od kogoś bliskie
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Co takiego dokładnie doprowadziło do tej wczorajszej kłótni? Jak się w
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Co dokładnie wtedy do Ciebie powiedział, ż

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 381


Pamięć tak naprawdę siedzi w naszym kodzie (w zmiennej historia), bo samo API LLM jest bezstanowe i nic nie pamięta. Z każdą wiadomością historia robi się coraz dłuższa i wysyłamy modelowi coraz większy blok tekstu, co w dłuższej perspektywie może zamulić działanie, podbić koszty i zjeść okno kontekstowe.

### ★ Zadanie 10d (dla chętnych) — fałszywa pamięć

Skoro pamięć to po prostu lista, którą **Ty** budujesz — możesz ją sfałszować.
Zbuduj historię, w której model rzekomo powiedział coś, czego nigdy nie
powiedział (wpisz własną wiadomość z rolą `assistant`), i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [ ]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 10d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

## 7. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach** — żeby zmienić rolę,
trzeba było napisać nowy skrypt. W LLM rolę ustala jeden akapit tekstu
w wiadomości `system`.

### 🔧 Zadanie 11
a) Dopisz do słownika `ROLE` **jedną własną rolę** i uruchom to samo zdanie
przez wszystkie role. Zwróć szczególną uwagę na rolę `"eliza"`: czy LLM
udający ELIZĘ łamie się w tych samych miejscach co prawdziwa ELIZA?

b) Uruchom **to samo** pytanie trzy razy przy dwóch temperaturach.
Porównaj losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się
w każdym z systemów?

In [18]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    "spongebob": ("Jestes spongebobem ktory zamiast kariery pod Tłustym Krabem poszedl studiowac psycologie,zwiezle"),
    # TODO (Zadanie 11a): dopisz własną rolę
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Rozumiem, to bardzo trudny stan. Chcesz o tym opowiedzieć?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Rozumiem Cię. Najlepiej przebyć samotność w pięknym otoczeniu, gdzie łatwo o inspirację – polecam **Księży Młyn**. Spacer zabytkowymi uliczkami i kawka w jednej z tutejszych klimatycznych kawiarni na pewno dadzą Ci poczucie spokoju.

[spongebob] *Śmieję się moim piskliwym, zaraźliwym śmiechem, machając dyplomem w dół*

Ojej, przyjacielu! *Wywalam oczy ze współczucia* Poczucie samotności to tak, jakby utknąć w słoiku z meduzami – boli, szczypie, a wokół nikogo! 

Ale spójrzmy na to z perspektywy psychodynamicznej. Samotność to często tylko wewnętrzny sygnał, że Twój "wewnętrzny Spongebob" potrzebuje teraz najwięcej czułości i uwagi od Ciebie samego! 

A co powiesz na eksperyment behawioralny? Zróbmy coś małego, co lubisz – zjedz Kraboburgera, puszczaj bańki mydlane albo... po prostu pogadaj ze mną! Jestem tu, gotowy do analizy (i przytulania)! 🧽💛

In [ ]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

In [17]:
# Zadanie 11b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

--- temperatura 0.0 ---
  To zupełnie naturalne, że odczuwasz teraz taki stres. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
  To zupełnie naturalne, że odczuwasz stres przed tak ważnym wydarzeniem. Co dokładnie dzieje się w Twojej głowie, kiedy leżysz w łóżku?
  To naturalne, że stres nie pozwala ci zasnąć, ale pamiętaj, że Twój mózg i tak odpoczywa, gdy leżysz z zamkniętymi oczami. Spróbuj wziąć kilka głębokich oddechów i skupić się na rozluźnieniu ciała.
--- temperatura 1.5 ---
  To zupełnie naturalne, że stres przed egzaminem utrudnia zasypianie. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
  To zrozumiałe, że czujesz stres, który nie pozwala Ci zasnąć. Spróbuj wziąć kilka głębokich oddechów i skupić się na ciele, zamiast na samych myślach o egzaminie.
  To naturalne, że stres nie pozwala ci zasnąć, ale pamiętaj, że Twój organizm ma już zgromadzoną wiedzę. Spróbuj wziąć kilka głębokich oddechów i skupić się na rozluźnieniu ciała.


**Obserwacje (Zadanie 11):**

a)0.0: Przy temperaturze 0.0 model generuje odpowiedzi w sposób deterministyczny i bardzo powtarzalny. Treści są stabilne, bezpieczne, ściśle trzymają się stonowanego, profesjonalnego kontekstu terapeutycznego i wykazują minimalne różnice między kolejnymi uruchomieniami

b)1.5: Wysoka temperatura 1.5 wprowadza dużą losowość i kreatywność do procesu generowania tekstu. Odpowiedzi znacznie bardziej się od siebie różnią pod względem doboru słownictwa, budowy zdań oraz proponowanych technik relaksacyjnych, co zwiększa ich różnorodność, ale niesie też ryzyko mniejszej przewidywalności.

## 8. Refleksja

### 🔧 Zadanie 12 (3–5 zdań na punkt)
1. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
2. W ELIZIE rola była zakodowana w regułach, w LLM to akapit tekstu (a historię
   rozmowy można sfałszować). Co to oznacza dla kogoś, kto buduje dziś
   „wirtualnego terapeutę"? Odwołaj się do obaw Weizenbauma z Lab 1.


**Twoje odpowiedzi (Zadanie 12):**

1. Które pęknięcia LLM naprawił, a które zostały? Dlaczego pamięć jest mechaniczna?
   LLM świetnie poradził sobie z sztywną gramatyką, odmianą zaimków oraz ogarnięciem ironii czy przeczeń. Został jednak problem z pamięcią – modele są bezstanowe, więc całą „pamięć” musimy sztucznie dopisywać w kodzie jako listę wiadomości do każdego strzału do API. Przez to działa to dość topornie i przy dłuższej gadce zapycha limit kontekstu.

2. Co zmiana roli na akapit tekstu oznacza dla twórców „wirtualnego terapeuty”?
   Skoro rola to tylko zwykły tekst, system jest mega podatny na kombinowanie i np. *prompt injection*. Nawiązując do obaw Weizenbauma, ludzie strasznie łatwo dają się nabrać na to, że maszyna „naprawdę” ich rozumie i czuje empatię (klasyczny efekt ELIZY). Tworzy to spory problem etyczny, bo użytkownik może powierzyć swoje realne problemy psychiczne bezdusznemu algorytmowi, który nie bierze za nic odpowiedzialności.

## Co oddajesz i jak oceniam

- **Kod (Zadania 8, 10a–b, 11a):** działa, wyniki są w notebooku.
- **Zadanie 9:** przewidywanie zapisane *przed* uruchomieniem + wypełniona tabela.
- **Zadanie 10c i 11:** trafne obserwacje z odwołaniem do kodu.
- **Zadanie 10d ★:** dodatkowe.
- **Zadanie 12:** samodzielna refleksja oparta na Twoich wynikach.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.